# GNN 消息传递与节点分类：从归一化邻接到推理合同

本 Notebook 用 NumPy、NetworkX 和 scikit-learn 拆开节点分类流水线：授权时态快照 → 节点特征 → `A+I` 对称归一化 → 0/1/多层消息传递 → 只在 train mask 上训练线性头 → validation 选层数 → test 一次性报告 → 版本化推理。所有数据均为固定随机种子的虚构服务节点，不下载数据。

这里的 `H^(k)=Â^k X` 加 LogisticRegression 是**可解释教学基线**，不是完整可训练 GCN、GraphSAGE，也不冒充 PyTorch Geometric/DGL 的生产实现。它足以暴露归一化、自环、泄漏、过平滑、同配/异配以及 transductive/inductive 边界。


## 1. 任务合同与威胁模型

目标是在指定 `tenant/as_of` 图快照上把节点分为 `frontend/backend`。模型输入只能含允许上线时获得的节点属性和图结构；训练损失只能读取 train 标签。若聚合时使用了 test 节点的无标签特征/边，这是 transductive 设定，必须明确披露；若声称 inductive，则测试节点及其边不能参与训练期表示计算。tenant 隔离必须在构造邻接矩阵前完成，否则即使最后隐藏节点，聚合向量仍泄漏结构。


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime, timezone
import hashlib
import json
import math

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, f1_score

RNG = np.random.default_rng(17)
FEATURE_ORDER = ("http_ratio", "batch_ratio", "cpu_norm", "bias")
MODEL_SCHEMA = "node-classifier-v1"

@dataclass(frozen=True)
class AuthContext:
    tenant: str
    scopes: frozenset[str]
    principal: str
    def require(self, scope: str):
        if scope not in self.scopes:
            raise PermissionError(scope)

def parse_time(value: str) -> datetime:
    parsed = datetime.fromisoformat(value.replace("Z", "+00:00"))
    return (parsed if parsed.tzinfo else parsed.replace(tzinfo=timezone.utc)).astimezone(timezone.utc)

def fingerprint(payload) -> str:
    raw = json.dumps(payload, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(raw.encode()).hexdigest()[:16]

auth_a = AuthContext("tenant-a", frozenset({"graph:read", "model:predict"}), "alice")
assert len(FEATURE_ORDER) == 4
assert parse_time("2026-07-01Z").tzinfo is not None


## 2. 构造受控图、特征和标签

tenant-a 有 36 个节点，两类各 18 个。同类节点形成局部环，并加入少量跨类边使整图连通；特征带可控噪声而非直接复制标签。tenant-b 的极端特征用于验证隔离。节点顺序通过稳定 ID 排序固定，不能依赖 NetworkX 插入顺序。标签只用于监督和离线评估，不写进图节点属性。


In [ ]:
graph_all = nx.Graph(schema_version="service-node-graph-v1")
features_by_node, labels_by_node = {}, {}
for i in range(36):
    cls = 0 if i < 18 else 1
    nid = f"tenant-a:n{i:02d}"
    base = np.array([1.3, 0.2, 0.45, 1.0]) if cls == 0 else np.array([0.2, 1.3, 0.65, 1.0])
    features_by_node[nid] = base + RNG.normal(0, [0.32, 0.32, 0.12, 0.0])
    labels_by_node[nid] = cls
    graph_all.add_node(nid, tenant="tenant-a", created_at="2026-01-01Z")
for offset in (0, 18):
    for j in range(18):
        u = f"tenant-a:n{offset+j:02d}"
        for step in (1, 2):
            v = f"tenant-a:n{offset+(j+step)%18:02d}"
            graph_all.add_edge(u, v, tenant="tenant-a", valid_from="2026-01-01Z", valid_to=None)
for j in (0, 6, 12):
    graph_all.add_edge(f"tenant-a:n{j:02d}", f"tenant-a:n{18+j:02d}", tenant="tenant-a", valid_from="2026-03-01Z", valid_to=None)
for j in range(4):
    nid = f"tenant-b:x{j}"; graph_all.add_node(nid, tenant="tenant-b", created_at="2026-01-01Z")
    features_by_node[nid] = np.array([99.0, 99.0, 99.0, 1.0]); labels_by_node[nid] = j % 2
for j in range(3):
    graph_all.add_edge(f"tenant-b:x{j}", f"tenant-b:x{j+1}", tenant="tenant-b", valid_from="2026-01-01Z", valid_to=None)

assert graph_all.number_of_nodes() == 40
assert set(labels_by_node) == set(features_by_node) == set(graph_all.nodes)
assert all(v.shape == (4,) and np.isfinite(v).all() for v in features_by_node.values())


## 3. 授权快照与 train/val/test mask

先过滤 tenant 与有效时间，再生成矩阵。每类 8 个 train、5 个 validation、5 个 test，保证分层且互斥。超参数（这里是聚合层数）只看 validation；test 在选型之后才打开。真实项目若节点随时间出现，应优先按时间切分，而不是随机拆分相邻节点。


In [ ]:
def authorized_snapshot(g: nx.Graph, auth: AuthContext, as_of: str) -> nx.Graph:
    auth.require("graph:read"); t = parse_time(as_of)
    nodes = [n for n, d in g.nodes(data=True) if d["tenant"] == auth.tenant and parse_time(d["created_at"]) <= t]
    out = nx.Graph(schema_version=g.graph["schema_version"], tenant=auth.tenant, as_of=as_of)
    out.add_nodes_from((n, dict(g.nodes[n])) for n in nodes)
    for u, v, d in g.edges(data=True):
        active = parse_time(d["valid_from"]) <= t and (d["valid_to"] is None or t < parse_time(d["valid_to"]))
        if u in out and v in out and d["tenant"] == auth.tenant and active:
            out.add_edge(u, v, **dict(d))
    return out

snapshot = authorized_snapshot(graph_all, auth_a, "2026-07-01Z")
nodes = sorted(snapshot.nodes()); X = np.vstack([features_by_node[n] for n in nodes]); y = np.array([labels_by_node[n] for n in nodes])
train_mask = np.zeros(len(nodes), bool); val_mask = np.zeros(len(nodes), bool); test_mask = np.zeros(len(nodes), bool)
for cls in (0, 1):
    idx = np.flatnonzero(y == cls); train_mask[idx[:8]] = True; val_mask[idx[8:13]] = True; test_mask[idx[13:]] = True
assert len(nodes) == 36 and not any(n.startswith("tenant-b") for n in nodes)
assert (train_mask.sum(), val_mask.sum(), test_mask.sum()) == (16, 10, 10)
assert np.all((train_mask.astype(int) + val_mask + test_mask) == 1)
assert set(y[train_mask]) == set(y[val_mask]) == set(y[test_mask]) == {0, 1}


## 4. `A + I` 与对称归一化

GCN 常用 `Â = D̃^{-1/2}(A+I)D̃^{-1/2}`。自环保留自身信息；归一化抑制度数差异造成的尺度爆炸。`Â` 对无向图应对称，但它通常不是逐行和为 1 的随机游走矩阵。孤立节点加自环后度数为 1，不会除零。注意：先全图归一化再切 tenant 仍会泄漏，顺序不能交换。


In [ ]:
def normalized_adjacency(g: nx.Graph, ordered_nodes: list[str]) -> np.ndarray:
    if set(ordered_nodes) != set(g.nodes) or len(ordered_nodes) != len(set(ordered_nodes)):
        raise ValueError("ordered_nodes 必须与授权快照一一对应")
    A = nx.to_numpy_array(g, nodelist=ordered_nodes, dtype=float, weight=None)
    A_tilde = A + np.eye(len(A)); degree = A_tilde.sum(axis=1)
    inv_sqrt = np.diag(1.0 / np.sqrt(degree))
    return inv_sqrt @ A_tilde @ inv_sqrt

A_hat = normalized_adjacency(snapshot, nodes)
assert A_hat.shape == (36, 36) and np.allclose(A_hat, A_hat.T)
assert np.all(np.diag(A_hat) > 0) and np.isfinite(A_hat).all()
unsafe_A = normalized_adjacency(graph_all, sorted(graph_all))  # 数学函数会成功，说明它本身不知道 auth
assert unsafe_A.shape == (40, 40) and A_hat.shape == (36, 36)
# 因而生产 API 必须把 authorized_snapshot 与 normalize 封装成不可绕过的同一入口。


## 5. 0/1/多层消息传递与过平滑

`H0=X` 是无图基线；`H1=ÂX` 聚合一跳；`Hk=Â^kX` 扩大感受野。深度增加并不免费：计算与邻居依赖扩大，节点表示可能趋同。下面用“各维方差均值”做简单过平滑代理；真实系统还应监控类间/类内距离、预测熵和分群性能。


In [ ]:
def propagate(A_norm: np.ndarray, X0: np.ndarray, depth: int) -> np.ndarray:
    if depth < 0 or int(depth) != depth:
        raise ValueError("depth 必须是非负整数")
    H = X0.copy()
    for _ in range(int(depth)):
        H = A_norm @ H
    return H

depths = [0, 1, 2, 4, 8, 16, 32]
embeddings = {d: propagate(A_hat, X, d) for d in depths}
dispersion = {d: float(np.var(H, axis=0).mean()) for d, H in embeddings.items()}
display(pd.DataFrame({"depth": depths, "feature_dispersion": [dispersion[d] for d in depths]}))
assert np.allclose(embeddings[0], X)
assert dispersion[32] < dispersion[0] * 0.35
assert all(np.isfinite(H).all() for H in embeddings.values())


## 6. 只在 train 标签上训练，用 validation 选层数

对每个深度都只拟合 `H[train_mask], y[train_mask]`。validation macro-F1 等权看待两类；若类别不平衡，还应同时报告每类召回、PR 曲线与业务成本。选出深度后只报告一次 test，不能反复看 test 调参。受控数据很小，分数只验证流水线，不代表泛化性能。


In [ ]:
candidate_depths = [0, 1, 2, 4, 8]
models, rows = {}, []
for depth in candidate_depths:
    H = embeddings[depth]
    model = LogisticRegression(C=1.0, max_iter=500, random_state=17).fit(H[train_mask], y[train_mask])
    models[depth] = model
    rows.append({"depth": depth,
                 "train_macro_f1": f1_score(y[train_mask], model.predict(H[train_mask]), average="macro"),
                 "val_macro_f1": f1_score(y[val_mask], model.predict(H[val_mask]), average="macro")})
scores = pd.DataFrame(rows); best_depth = int(scores.sort_values(["val_macro_f1", "depth"], ascending=[False, True]).iloc[0]["depth"])
best_model, H_best = models[best_depth], embeddings[best_depth]
test_pred = best_model.predict(H_best[test_mask]); test_macro_f1 = f1_score(y[test_mask], test_pred, average="macro")
display(scores.round(3)); print("selected_depth=", best_depth, "test_macro_f1=", round(test_macro_f1, 3))
display(pd.DataFrame(confusion_matrix(y[test_mask], test_pred), index=["true_front", "true_back"], columns=["pred_front", "pred_back"]))
assert best_depth in candidate_depths and 0 <= test_macro_f1 <= 1
assert all(models[d].n_features_in_ == X.shape[1] for d in candidate_depths)


### 6.1 为什么当前结果选择了 depth=0？

固定种子下，各候选深度的 validation macro-F1 都达到 1.0；稳定 tie-break 因而选择复杂度最低的 `depth=0`。这意味着当前受控数据只证明了评估与选型链路有效，**没有证明图传播带来增益**：原始节点属性已经足够分离两类。真实验收应报告相对 0-hop 基线的增益、置信区间和多时间切片结果；若图模型没有稳定增益，应上线更简单的无图分类器，而不是为了模型名称强行选择 GNN。


## 7. 失败反例：把标签或未来统计写进特征

若 `target_label`、由全量标签计算的“同类邻居比例”、事后人工处置状态进入 X，离线分数会虚高。另一个隐蔽泄漏是先在全体节点上拟合标准化/特征选择，再拆分。字段名黑名单只能作为第一层低成本防线，无法识别改名后的泄漏字段；因此还要绑定特征来源、最大事件时间、预处理 fit split、监督 fit split 和表示范围，并由独立数据血缘系统核验，而不能只相信调用者自报的 metadata。


In [ ]:
FORBIDDEN_FEATURES = {"label", "target_label", "post_incident_resolution", "future_degree"}
FEATURE_LINEAGE = {"feature_order": list(FEATURE_ORDER), "source_snapshot": "synthetic-node-features-v1",
                   "max_event_time": "2026-06-30Z", "preprocessing_fit_split": "not_applicable",
                   "supervised_fit_split": "train_only",
                   "representation_scope": "authorized_transductive_snapshot_without_labels"}
FEATURE_SCHEMA_ID = fingerprint(FEATURE_LINEAGE)

def validate_feature_contract(names, matrix, lineage, prediction_as_of):
    overlap = FORBIDDEN_FEATURES.intersection(names)
    if overlap:
        raise ValueError(f"发现泄漏字段: {sorted(overlap)}")
    if matrix.shape[1] != len(names) or not np.isfinite(matrix).all():
        raise ValueError("特征形状或有限性不满足合同")
    if lineage.get("feature_order") != list(names):
        raise ValueError("lineage 中的特征顺序不匹配")
    if parse_time(lineage.get("max_event_time", "9999-12-31Z")) > parse_time(prediction_as_of):
        raise ValueError("特征事件时间晚于预测快照")
    if lineage.get("preprocessing_fit_split") not in {"train_only", "not_applicable"}:
        raise ValueError("预处理器不是仅在 train 上拟合")
    if lineage.get("supervised_fit_split") != "train_only":
        raise ValueError("监督模型读取了 train 以外标签")
    if lineage.get("representation_scope") != "authorized_transductive_snapshot_without_labels":
        raise ValueError("表示计算范围不符合披露的 transductive 合同")
    return True

assert validate_feature_contract(FEATURE_ORDER, X, FEATURE_LINEAGE, snapshot.graph["as_of"])
try:
    validate_feature_contract(FEATURE_ORDER + ("target_label",), np.c_[X, y], FEATURE_LINEAGE, snapshot.graph["as_of"])
    raise AssertionError("标签泄漏未被拒绝")
except ValueError as exc:
    assert "target_label" in str(exc)
for bad_patch in ({"max_event_time": "2026-08-01Z"}, {"preprocessing_fit_split": "all_nodes"}):
    bad_lineage = {**FEATURE_LINEAGE, **bad_patch}
    try:
        validate_feature_contract(FEATURE_ORDER, X, bad_lineage, snapshot.graph["as_of"])
        raise AssertionError("非法 lineage 未被拒绝")
    except ValueError:
        pass


## 8. 同配与异配：邻居不总是“同类证据”

edge homophily 是同标签边占比，只能离线用已标注样本估计，不能作为在线特征偷看标签。本受控图同配较强，因此平滑可能有益；在买卖、对抗、二部关系等异配图中，邻居常属于不同类，朴素平均会冲淡甚至反转信号。模型选择要先看关系语义，而不是默认堆 GCN 层。


In [ ]:
same = sum(labels_by_node[u] == labels_by_node[v] for u, v in snapshot.edges())
edge_homophily = same / snapshot.number_of_edges()
hetero = nx.Graph(); hetero.add_nodes_from(nodes)
for i in range(18):
    for step in (0, 1, 2):
        hetero.add_edge(nodes[i], nodes[18 + (i + step) % 18])
A_hetero = normalized_adjacency(hetero, nodes); H_hetero = A_hetero @ X
def centroid_gap(H):
    return float(np.linalg.norm(H[y == 0].mean(axis=0) - H[y == 1].mean(axis=0)))
print({"homophily": round(edge_homophily, 3), "raw_gap": round(centroid_gap(X), 3), "hetero_1hop_gap": round(centroid_gap(H_hetero), 3)})
assert edge_homophily > 0.9
assert centroid_gap(H_hetero) < centroid_gap(X)


## 9. Transductive 与 inductive 的边界

当前 `Â` 包含 val/test 节点及其边，但训练没有读取其标签，因此是典型 transductive 半监督设定。它不能声称能处理训练时完全未见的新节点。GraphSAGE 的核心思想是学习可复用的邻居聚合函数，从属性生成未见节点表示；但采样、可学习权重和多层非线性都不在本例中。下面仅演示“训练子图邻接”和“新节点均值聚合”的合同差别。


In [ ]:
train_nodes = [n for n, keep in zip(nodes, train_mask) if keep]
train_subgraph = snapshot.subgraph(train_nodes).copy()
A_train = normalized_adjacency(train_subgraph, train_nodes)
assert A_train.shape == (16, 16) and set(train_subgraph).isdisjoint({n for n, keep in zip(nodes, test_mask) if keep})

def inductive_mean_demo(new_features: np.ndarray, neighbor_features: np.ndarray) -> np.ndarray:
    if new_features.shape != (len(FEATURE_ORDER),) or neighbor_features.ndim != 2 or neighbor_features.shape[1] != len(FEATURE_ORDER):
        raise ValueError("特征合同不匹配")
    if len(neighbor_features) == 0:
        return new_features.copy()  # cold-start fallback
    return np.vstack([new_features, neighbor_features]).mean(axis=0)

new_embedding = inductive_mean_demo(np.array([1.0, 0.2, 0.5, 1.0]), X[train_mask][:2])
assert new_embedding.shape == (4,) and np.isfinite(new_embedding).all()


## 10. 模型制品与数值推理合同

制品至少绑定：模型/图 schema、特征顺序与血缘、聚合深度、训练快照、类别映射、线性头参数、代码版本、图指纹和表示快照 ID。模型参数与节点表示必须作为一个不可拆错的 bundle 校验；下面把二分类 LogisticRegression 参数导出为纯 NumPy 可复核形式，并为制品与表示分别计算内容哈希。真实系统应使用带签名的模型注册表与不可变对象存储。


In [ ]:
def graph_payload_for(g: nx.Graph, ordered_nodes: list[str]) -> dict:
    if set(ordered_nodes) != set(g.nodes) or ordered_nodes != sorted(ordered_nodes):
        raise ValueError("图指纹要求完整且稳定排序的节点列表")
    return {"nodes": ordered_nodes, "edges": sorted(tuple(sorted((u, v))) for u, v in g.edges())}

graph_payload = graph_payload_for(snapshot, nodes); graph_fingerprint = fingerprint(graph_payload)
representation_snapshot = {"representation_schema": "node-representation-v1", "tenant": auth_a.tenant,
                           "as_of": snapshot.graph["as_of"], "graph_fingerprint": graph_fingerprint,
                           "depth": best_depth, "feature_order": list(FEATURE_ORDER),
                           "feature_schema_id": FEATURE_SCHEMA_ID, "feature_lineage": FEATURE_LINEAGE,
                           "node_ids": nodes, "matrix": H_best.tolist()}
def representation_digest(value: dict) -> str:
    return fingerprint({k: v for k, v in value.items() if k != "representation_id"})
representation_snapshot["representation_id"] = representation_digest(representation_snapshot)

artifact = {"model_schema": MODEL_SCHEMA, "graph_schema": snapshot.graph["schema_version"],
            "code_version": "g14-notebook-v2", "tenant": auth_a.tenant, "as_of": snapshot.graph["as_of"],
            "feature_order": list(FEATURE_ORDER), "feature_schema_id": FEATURE_SCHEMA_ID,
            "aggregation": "symmetric-normalized-adjacency-with-self-loop", "depth": best_depth,
            "classes": best_model.classes_.tolist(), "coef": best_model.coef_.tolist(),
            "intercept": best_model.intercept_.tolist(), "graph_fingerprint": graph_fingerprint,
            "representation_id": representation_snapshot["representation_id"], "training_seed": 17}
def artifact_digest(value: dict) -> str:
    return fingerprint({k: v for k, v in value.items() if k != "artifact_id"})
artifact["artifact_id"] = artifact_digest(artifact)

def artifact_predict_proba(H: np.ndarray, artifact: dict) -> np.ndarray:
    coef, intercept = np.asarray(artifact["coef"])[0], float(artifact["intercept"][0])
    logit = np.clip(H @ coef + intercept, -40, 40); p1 = 1.0 / (1.0 + np.exp(-logit))
    return np.c_[1 - p1, p1]

manual_proba = artifact_predict_proba(H_best, artifact)
assert np.allclose(manual_proba, best_model.predict_proba(H_best), atol=1e-10)
assert artifact_digest(artifact) == artifact["artifact_id"] and representation_digest(representation_snapshot) == representation_snapshot["representation_id"]
assert artifact["feature_order"] == list(FEATURE_ORDER) and len(artifact["artifact_id"]) == 16


## 11. 推理服务：已知节点与新节点不能混为一谈

该制品是绑定快照的 transductive 模型，因此只接受制品索引中的节点；新节点必须走明确的 inductive 模型或 cold-start 回退。客户端不能覆盖 tenant、特征顺序、聚合深度和图版本。服务应从内部授权图快照计算指纹，并同时验证 artifact 内容哈希、representation 内容哈希及二者的 tenant/as-of/graph/depth/feature-schema 绑定。输出应带 artifact 与 representation ID、概率、拒绝原因与表示来源，且日志不写原始敏感属性。


In [ ]:
def validate_model_bundle(model_artifact: dict, representation: dict, current_graph: nx.Graph) -> bool:
    if artifact_digest(model_artifact) != model_artifact.get("artifact_id"):
        raise ValueError("artifact 内容哈希不匹配")
    if representation_digest(representation) != representation.get("representation_id"):
        raise ValueError("representation 内容哈希不匹配")
    if model_artifact.get("representation_id") != representation.get("representation_id"):
        raise ValueError("artifact 未绑定当前 representation")
    for field in ("tenant", "as_of", "graph_fingerprint", "depth", "feature_schema_id"):
        if model_artifact.get(field) != representation.get(field):
            raise ValueError(f"模型与表示的 {field} 不匹配")
    current_nodes = sorted(current_graph.nodes())
    current_fingerprint = fingerprint(graph_payload_for(current_graph, current_nodes))
    if current_graph.graph.get("tenant") != model_artifact["tenant"] or current_graph.graph.get("as_of") != model_artifact["as_of"] or current_fingerprint != model_artifact["graph_fingerprint"]:
        raise ValueError("当前授权图快照与制品不匹配")
    if tuple(model_artifact.get("feature_order", ())) != FEATURE_ORDER or fingerprint(representation.get("feature_lineage")) != model_artifact["feature_schema_id"]:
        raise ValueError("特征 schema 或 lineage 不匹配")
    matrix = np.asarray(representation["matrix"], dtype=float)
    validate_feature_contract(FEATURE_ORDER, matrix, representation["feature_lineage"], model_artifact["as_of"])
    if representation.get("node_ids") != current_nodes or matrix.shape != (len(current_nodes), len(FEATURE_ORDER)):
        raise ValueError("representation 节点索引或形状不匹配")
    return True

def predict_known_node(auth: AuthContext, node: str, model_artifact: dict, representation: dict, current_graph: nx.Graph):
    auth.require("model:predict")
    if auth.tenant != model_artifact.get("tenant") or current_graph.graph.get("tenant") != auth.tenant:
        raise PermissionError("tenant 不匹配")
    validate_model_bundle(model_artifact, representation, current_graph)
    node_to_row = {n: i for i, n in enumerate(representation["node_ids"])}
    if node not in node_to_row:
        raise KeyError("未知节点：该 transductive 制品不支持 cold start")
    embedding = np.asarray(representation["matrix"], dtype=float)[node_to_row[node]]
    probs = artifact_predict_proba(embedding[None, :], model_artifact)[0]; winner = int(np.argmax(probs))
    return {"node_id": node, "class_id": int(model_artifact["classes"][winner]), "probabilities": probs.tolist(),
            "artifact_id": model_artifact["artifact_id"], "representation_id": representation["representation_id"],
            "representation": f"snapshot-depth-{model_artifact['depth']}"}

prediction = predict_known_node(auth_a, nodes[0], artifact, representation_snapshot, snapshot)
assert abs(sum(prediction["probabilities"]) - 1) < 1e-12
tamper_cases = {"artifact_id": "forged", "depth": best_depth + 1, "as_of": "2026-07-02Z",
                "graph_fingerprint": "stale-graph", "feature_schema_id": "stale-schema"}
for field, bad_value in tamper_cases.items():
    forged = dict(artifact); forged[field] = bad_value
    if field != "artifact_id":
        forged["artifact_id"] = artifact_digest(forged)  # 即使攻击者重算单体 hash，跨制品绑定仍应失败
    try:
        predict_known_node(auth_a, nodes[0], forged, representation_snapshot, snapshot)
        raise AssertionError(f"伪造字段未被拒绝: {field}")
    except ValueError:
        pass
try:
    predict_known_node(AuthContext("tenant-b", frozenset({"model:predict"}), "mallory"), nodes[0], artifact, representation_snapshot, snapshot)
    raise AssertionError("跨 tenant 推理未拒绝")
except PermissionError:
    pass


## 12. 可观测、漂移与生产替换点

离线记录每个 split 的样本数/类别分布、图指纹、度分布、同配率、各深度 macro-F1、混淆矩阵和随机种子。在线监控输入缺失率、特征分布、节点度/cold-start 比例、预测概率/熵、拒绝率、延迟和 tenant 越权计数；标签延迟到达后再计算真实 F1。高基数 node ID 不作监控标签。

生产替换点包括 PyTorch Geometric/DGL 的可学习 GCN/GraphSAGE、邻居采样、mini-batch、分布式特征存储、时间感知图、模型注册表、shadow/canary 与回滚。替换后仍应保留这里的快照、mask、泄漏、权限、版本和 cold-start 合同。


In [ ]:
def monitoring_snapshot(g: nx.Graph, H: np.ndarray, probabilities: np.ndarray) -> dict:
    entropy = -np.sum(np.clip(probabilities, 1e-12, 1) * np.log(np.clip(probabilities, 1e-12, 1)), axis=1)
    degrees = np.array([g.degree(n) for n in nodes], dtype=float)
    return {"node_count": len(g), "edge_count": g.number_of_edges(), "degree_mean": float(degrees.mean()),
            "isolated_rate": float(np.mean(degrees == 0)), "feature_nonfinite": int((~np.isfinite(H)).sum()),
            "prediction_entropy_mean": float(entropy.mean()), "graph_fingerprint": artifact["graph_fingerprint"]}

monitor = monitoring_snapshot(snapshot, H_best, manual_proba)
display(monitor)
assert monitor["node_count"] == 36 and monitor["feature_nonfinite"] == 0
assert 0 <= monitor["prediction_entropy_mean"] <= math.log(2) + 1e-12
assert artifact_digest(artifact) == artifact["artifact_id"]
assert representation_digest(representation_snapshot) == artifact["representation_id"]


## 13. 验收清单与失败策略

1. 快照先做 tenant/time 过滤；2. 节点顺序、特征顺序和类别映射固定；3. mask 互斥且 loss 只看 train；4. validation 选型、test 单次报告；5. 0-hop 是必须基线；6. 报告过平滑与同/异配假设；7. transductive 不包装成 inductive；8. 制品绑定图指纹与 as_of；9. 未知节点、缺失特征、越权、图版本不匹配均 fail closed；10. 线上退化时回退到无图分类器或人工规则，并产生告警。


In [ ]:
# 最终契约测试
assert set(artifact) >= {"artifact_id", "representation_id", "graph_fingerprint", "feature_schema_id", "feature_order", "depth", "as_of", "tenant"}
assert len(set(nodes)) == len(nodes) and nodes == sorted(nodes)
assert not np.shares_memory(embeddings[0], X)
assert prediction["node_id"] in snapshot and prediction["artifact_id"] == artifact["artifact_id"]
assert prediction["representation_id"] == representation_snapshot["representation_id"] and validate_model_bundle(artifact, representation_snapshot, snapshot)
print("GNN 消息传递教学流水线验收通过；这不是生产 GNN 框架基准。")


## 14. 原始资料

- Kipf & Welling, *Semi-Supervised Classification with Graph Convolutional Networks*：https://arxiv.org/abs/1609.02907
- Hamilton, Ying & Leskovec, *Inductive Representation Learning on Large Graphs (GraphSAGE)*：https://proceedings.neurips.cc/paper/2017/hash/5dd9db5e033da9c6fb5ba83c7a7ebea9-Abstract.html
- NetworkX 图类型与邻接矩阵官方文档：https://networkx.org/documentation/stable/reference/classes/ 与 https://networkx.org/documentation/stable/reference/generated/networkx.convert_matrix.to_numpy_array.html

论文定义了模型思想；本 Notebook 重点补足数据泄漏、权限、时态快照、制品和可观测工程合同。受控合成数据上的结果不可外推到真实图。
